In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torch.utils.data import DataLoader, TensorDataset

import torch.optim as optim
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models.signature import infer_signature

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_colwidth', None)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy', 'query_embedding.npy']

data = []
for file in files:
    data.append(np.load(f'../data/new_embeddings/{file}'))

In [5]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4],data[5]),axis=1)

In [6]:
labels = pd.read_csv(f'../data/data.csv')['binary_label'].values

In [7]:
embedding.shape

(99909, 4608)

In [8]:
labels.shape

(99909,)

In [9]:
x_train, x_test, y_train, y_test = train_test_split(embedding, labels, test_size=0.2, random_state=42, shuffle=True)

In [10]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [11]:
product_idx = 3840

In [12]:
# query_embeddings = x_train[:79904,product_idx:]
# product_embeddings = x_train[:79904,:product_idx]

In [13]:
query_embeddings.shape

NameError: name 'query_embeddings' is not defined

In [14]:
class QueryTower(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QueryTower, self).__init__()
        self.query_tower = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, query_input):
        query_emb = self.query_tower(query_input)
        query_emb = nn.functional.normalize(query_emb, p=2, dim=1)
        
        return query_emb


In [15]:
class ProductTower(nn.Module):    
    def __init__(self, input_dim, output_dim):
        super(ProductTower, self).__init__()
        self.product_tower = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, product_input):
        product_emb = self.product_tower(product_input)
        product_emb = nn.functional.normalize(product_emb, p=2, dim=1)
    
        return product_emb
    

In [16]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, query_emb, pos_product_emb, neg_product_emb):
        pos_dist = torch.norm(query_emb - pos_product_emb, p=2, dim=1)
        neg_dist = torch.norm(query_emb - neg_product_emb, p=2, dim=1)
        # print(pos_dist.shape, neg_dist.shape)
        loss = torch.mean(torch.relu(pos_dist - neg_dist + self.margin))
        # print(loss)
        return loss

In [17]:
query_dim = 768
product_dim = 3840

output_dim = 32
batch_size = 32

train_size = x_train.shape[0] - (x_train.shape[0] % 32)
num_epochs = 50
test_size = x_test.shape[0] - (x_test.shape[0] % 32)

train_num_batches = int(train_size/batch_size)
test_num_batches = int(test_size/batch_size)

query_model = QueryTower(query_dim, output_dim).to(device)
product_model = ProductTower(product_dim, output_dim).to(device)
optimizer = optim.Adam(list(query_model.parameters()) + list(product_model.parameters()), lr=0.001)
criterion = ContrastiveLoss()

train_dataset = TensorDataset(x_train[:train_size,product_idx:], x_train[:train_size,:product_idx])
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(x_test[:test_size,product_idx:], x_test[:test_size,:product_idx])
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [18]:
train_size

79904

In [19]:
mlflow.set_tracking_uri(uri='http://127.0.0.1:5050')
mlflow.start_run()
mlflow.log_param("epochs", num_epochs)

loss = 0

# Training loop
for epoch in tqdm(range(num_epochs)):    

    query_model.train()
    product_model.train()
    train_batch_loss = 0

    for query_batch, pos_product_batch in train_dataloader:

        query_batch = query_batch.to(device)
        pos_product_batch = pos_product_batch.to(device)

        # Negative sampling: shuffle product embeddings
        neg_product_batch = x_train[:train_size,:product_idx][torch.randperm(train_size)[:batch_size]]
        neg_product_batch = neg_product_batch.to(device)

        # print(neg_product_batch.shape)

        # Forward pass
        query_emb = query_model(query_batch)
        pos_product_emb = product_model(pos_product_batch)
        neg_product_emb = product_model(neg_product_batch)

        # print(query_emb.shape, pos_product_emb.shape, neg_product_emb.shape)
        # Compute loss
        train_batch_loss = criterion(query_emb, pos_product_emb, neg_product_emb)

        # Backpropagation
        optimizer.zero_grad()
        train_batch_loss.backward()
        optimizer.step()

        train_batch_loss += train_batch_loss

    total_train_loss = train_batch_loss/train_num_batches

    query_model.eval()
    product_model.eval()

    test_batch_loss = 0
    with torch.no_grad():
        for query_batch, product_batch in test_dataloader:

            query_batch = query_batch.to(device)
            product_batch = product_batch.to(device)


            neg_product_batch = x_test[:test_size,:product_idx][torch.randperm(test_size)[:batch_size]]
            neg_product_batch = neg_product_batch.to(device)

            query_emb = query_model(query_batch)
            pos_product_emb = product_model(pos_product_batch)
            neg_product_emb = product_model(neg_product_batch)

            test_batch_loss = criterion(query_emb, pos_product_emb, neg_product_emb)

            test_batch_loss += test_batch_loss

    total_test_loss = test_batch_loss/test_num_batches

    if (epoch+1) % 10 == 0:    
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_train_loss:.4f}, Test Loss: {total_test_loss:.4f}")


mlflow.log_metric("Average Loss", total_test_loss)
mlflow.pytorch.log_model(product_model, 'Product Tower', signature=infer_signature(torch.rand([1,product_dim])))
mlflow.pytorch.log_model(query_model, 'Query Tower', signature=infer_signature(torch.rand([1,query_dim])))

mlflow.end_run()


 20%|██        | 10/50 [02:12<08:52, 13.32s/it]

Epoch 10/50, Train Loss: 0.0001, Test Loss: 0.0035


 40%|████      | 20/50 [04:24<06:41, 13.38s/it]

Epoch 20/50, Train Loss: 0.0001, Test Loss: 0.0031


 60%|██████    | 30/50 [06:37<04:27, 13.38s/it]

...

Epoch 40/50, Train Loss: 0.0001, Test Loss: 0.0035


100%|██████████| 50/50 [10:53<00:00, 13.06s/it]
2025/04/02 17:36:26 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.


Epoch 50/50, Train Loss: 0.0001, Test Loss: 0.0032


2025/04/02 17:36:29 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.


🏃 View run sassy-bird-429 at: http://127.0.0.1:5050/#/experiments/0/runs/39921eca67b74c34b0c51f98e4526f5f
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


In [49]:
mlflow.end_run()


🏃 View run trusting-moth-515 at: http://127.0.0.1:5050/#/experiments/0/runs/170eb33701eb4a45aeb2ef3862130f9c
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


In [20]:
embedding = torch.from_numpy(embedding)

In [22]:
import faiss

product_model.eval()
with torch.no_grad():
    product_embeddings_faiss = product_model(embedding[:,:3840].to(device))

In [24]:
index = faiss.IndexFlatL2(32)

index.add(product_embeddings_faiss.cpu().numpy())

In [37]:
def retrieve_products(query, top_k=5):
    query_emb = query_model(query.to(device))
    query_emb_np = query_emb.cpu().detach().numpy()
    
    _, indices = index.search(query_emb_np, top_k) 
    return indices

In [26]:
df = pd.read_csv('../data/data.csv')

In [27]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base',trust_remote_code=True)


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: {'classifier.bias', 'classifier.weight'}
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [28]:
def get_embeddings(text):
    embeddings = emb_model.encode(text, normalize_embeddings=True, show_progress_bar=True)
    return embeddings.squeeze()


In [29]:
emb_example = torch.from_numpy(get_embeddings(df.iloc[100].to_list()[1:7]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
emb_example = emb_example.reshape((1,4608))

In [31]:
new_query = emb_example[:,product_idx:]
product = emb_example[:,:product_idx]

In [38]:
recommended_products = retrieve_products(new_query, top_k=5)
print("Recommended Product Indices:", recommended_products)


Recommended Product Indices: [[28094   100 51588 23565 50833]]


In [39]:
recommended_products[0]

array([28094,   100, 51588, 23565, 50833])

In [40]:
df.iloc[100]

product_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [41]:
df.iloc[recommended_products[0]]

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,query,esci_label,split,binary_label
28094,B089GYKNPL,2pcs fashion couple rings stainless ring steel rotatable bottle opener party ring creative versatile beer bottle openershipment from usa fast delivery blakslive 7,punk wedding mens stainless band ring spin titanium steel rotatable chain product description this is a cool chaintype rotating ring that can be used as a bottle opener you can perform magic tricks to open the beer smoothly at the party this is very enviable and very interesting but also makes us a party star stylish design exquisite fashion technical index material perfect titanium steel product weight 6g note due to the different display and lighting effects the actual color of the product may be slightly different from the color shown on the picture,fashion beer bottle openereasy to open the bottle it can make you in the party the field the bar the school and other gatherings with friends make you quick and quick beer bottle so you quickly become a friends eye magician give your friends a deep impression\nmade of highquality stainless steel durable the inside and surface are well polished and comfortable to wear it can withstand prolonged wear and tear\nthis ring has an inner ring that can be rotated manually and locked in the main chain this is a very interesting ring a stylish design suitable for everyday use and very suitable as a tool to relieve stress especially for manic or irritable people\nunique and fashionable designa meaningful way to share lovevery suitable for the gift selection of special occasions such as christmas valentines day birthday anniversary wedding very suitable for loversyou can also think of it as a friendship ring one for you and the other for your friends,na,blakslive,bottle opener ring,E,train,1
100,B07V35Y7M2,10 pieces ring bottle opener stainless steel beer bottle opener colorful finger bottle opener for party family gift supplies,features practical size every beautiful and practical ring bottle opener is only 22 mm 087 inches it fits on the finger very well which makes it easier to open the bottle portable and convenient to carry sturdy material these ring bottle openers are made of quality stainless steel and they are extremely sturdy and durable they help you open every bottle faster and safer can last for a long time use stylish design all ring bottle openers are designed to be very stylish keeping up with the trend of the times bright colors will make them more popular with everyone bring you good mood when using it specifications material stainless steel color as pictures shown size 22 mm 087 inches package includes 10 x ring bottle openers,practical size every beautiful and practical ring bottle opener is only 22 mm 087 inches it fits on the finger very well which makes it easier to open the bottle portable and convenient to carry\nsturdy material these ring bottle openers are made of quality stainless steel and they are extremely sturdy and durable they help you open every bottle faster and safer can last for a long time use\nrich variety there are five different colors ring bottle openers in each bag and each comes with two colors there are a wide variety of choices for you to choose from which is definitely enough for your daily life color as pictures shown\ndiverse usage you can use these ring bottle openers to open the bottles at the party or you can give them as a gift to your family or friends to better promote your relationship it will definitely be a very popular gift\nstylish design all ring bottle openers are designed to be very stylish keeping up with the trend of the times bright colors will make them more popular with everyone bring you good mood when using it,boao,as pictures shown,bottle opener ring,E,train,1
51588,B074FXTH6K,vqysko bottle opener ring bandbeer bar tool creative versatile stainless steel finger party ring silver 8,vqysko beer bar tool creative versa